# ENTSO-E clean fetch (DE/LU)
Load forecast, thermal generation/capacity/utilization, net import, activated aFRR/mFRR-up.


ENTSO-E data was manually checked and validated on their [website](https://transparency.entsoe.eu/load/total/dayAhead?appState=%7B%22sa%22%3A%5B%22BZN%7C10Y1001A1001A82H%22%5D%2C%22st%22%3A%22BZN%22%2C%22mm%22%3Atrue%2C%22ma%22%3Afalse%2C%22sp%22%3A%22HALF%22%2C%22dt%22%3A%22TABLE%22%2C%22df%22%3A%5B%222022-01-01%22%2C%222022-01-01%22%5D%2C%22tz%22%3A%22CET%22%7D) as of 27.01.2026.

In [7]:
# Undo Timedelta patch to allow default behavior (warning may appear)
import pandas as pd
if hasattr(pd, '_orig_Timedelta'):
    pd.Timedelta = pd._orig_Timedelta
import warnings


In [8]:
from datetime import datetime, timezone
import importlib
import pandas as pd
import energy_trading.ingestion.fetch_entsoe as fe

# Ensure notebook uses latest local module code (avoids stale kernel imports).
importlib.reload(fe)

BIDDING_ZONE = fe.BIDDING_ZONE
start = datetime(2022, 6, 20, 0, 0, tzinfo=timezone.utc)
end = datetime(2022, 7, 10, 0, 0, tzinfo=timezone.utc)

# Use the same API calls as the ingestion script.
entsoe_pl = fe.fetch_and_merge(
    start.strftime('%Y%m%d%H%M'),
    end.strftime('%Y%m%d%H%M'),
    BIDDING_ZONE,
    timeout=120,
)
entsoe_df = entsoe_pl.to_pandas()

# Convenience helpers
def col_or_empty(name):
    if name in entsoe_df.columns:
        s = entsoe_df.set_index('timestamp_utc')[name].copy()
        s.index = pd.to_datetime(s.index, utc=True)
        return s
    return pd.Series(dtype=float)

load_forecast = col_or_empty('system_load_forecast')
load_forecast.name = 'Ltfc'

gen_thermal = col_or_empty('GEN_THERMAL')
cap_thermal = col_or_empty('CAP_THERMAL')
util_thermal = col_or_empty('U_THERMAL')

EtaFRR_up = col_or_empty('activated_balancing_quantities_affr')
EtmFRR_up = col_or_empty('activated_balancing_quantities_mffr')


Attempt 1 failed: 
Attempt 2 failed: 
Attempt 3 failed: 
Installed capacity wind_onshore empty for 2022.
Installed capacity wind_offshore empty for 2022.


In [9]:
import os

start = pd.Timestamp('2022-01-01', tz='UTC')
end   = pd.Timestamp('2025-12-31', tz='UTC')
thermal_types = [
    'Fossil Brown coal/Lignite','Fossil Coal-derived gas','Fossil Gas',
    'Fossil Hard coal','Fossil Oil','Fossil Oil shale','Fossil Peat',
    'Nuclear','Waste','Other'
]
neighbors = ['10YAT-APG------L','10YBE----------2','10YCH-SWISSGRIDZ','10YCZ-CEPS-----N',
             '10YDK-1--------W','10YDK-2--------M','10YFR-RTE------C','10YNL----------L',
             '10YNO-2--------T','10YPL-AREA-----S','10Y1001A1001A47J']

# Client for direct exploratory ENTSO-E API calls used below.
api_key = os.getenv('ENTSOE_API_TOKEN') or os.getenv('ENTSOE_API_KEY')
if not api_key:
    raise RuntimeError('Missing ENTSOE_API_TOKEN (or ENTSOE_API_KEY) in environment.')
client = fe.EntsoePandasClient(api_key=api_key)


In [10]:
# Load forecast (use entsoe-py, fallback A01->A31)
try:
    load_forecast = client.query_load_forecast(BIDDING_ZONE, start=start, end=end, process_type='A01')
except Exception as e:
    print('A01 failed, trying A31:', e)
    load_forecast = client.query_load_forecast(BIDDING_ZONE, start=start, end=end, process_type='A31')
load_forecast.name = 'Ltfc'

# Actual thermal generation (per type, sum thermal PSR)
gen_per_type = client.query_generation(BIDDING_ZONE, start=start, end=end, psr_type=None)
thermal_cols = [t for t in thermal_types if t in gen_per_type.columns]
gen_thermal = gen_per_type[thermal_cols].sum(axis=1).rename('GEN_thermal')

# Installed thermal capacity
installed_per_type = client.query_installed_generation_capacity(
    BIDDING_ZONE, start=start, end=end, psr_type=None
)
cap_cols = [t for t in thermal_types if t in installed_per_type.columns]
cap_thermal = installed_per_type[cap_cols].sum(axis=1).rename('CAP_thermal')

util_thermal = (gen_thermal / cap_thermal).rename('UT_thermal')


A01 failed, trying A31: name 'client' is not defined


NameError: name 'client' is not defined

In [ ]:
# Net import (sum of neighbor imports minus exports)
def flow_pair(zone_in, zone_out):
    return client.query_crossborder_flows(zone_in, zone_out, start=start, end=end)
flows = []
for n in neighbors:
    f_imp = flow_pair(n, BIDDING_ZONE)
    f_exp = flow_pair(BIDDING_ZONE, n)
    flows.append(f_imp - f_exp)
Xtnet = pd.concat(flows, axis=1).sum(axis=1).rename('Xtnet')


HERE


In [ ]:
data_path = Path('../../data/entsoe_clean.parquet')
data_path.parent.mkdir(parents=True, exist_ok=True)
data.to_parquet(data_path, compression='zstd')
data.describe()


<hr>

In [ ]:
import requests
import os

ENTSOE_API_KEY = os.getenv('ENTSOE_API_KEY') 
base_url = f"https://web-api.tp.entsoe.eu/api?securityToken={ENTSOE_API_KEY}"

start = "202201010000"
end = "202301010000"
BiddingZoneDELU = "10Y1001A1001A82H"
processType_aFRR = "A51" 

headers = {}
payload = {}

# ==============================================================================
# 1. CAPACITY PRICE (Hier MUSS es "Area_Domain" heißen)
# ==============================================================================
print("Lade aFRR Leistungspreise (Capacity)...")
url_capacity = (f"{base_url}&periodStart={start}&periodEnd={end}"
                f"&documentType=A15&processType={processType_aFRR}"
                f"&type_MarketAgreement.Type=A01"
                f"&Area_Domain={BiddingZoneDELU}") # <--- Area_Domain

response_capacity = requests.request("GET", url_capacity, headers=headers, data=payload)

if response_capacity.status_code == 200:
    print("✅ Capacity erfolgreich!")
    with open("afrr_capacity_2022.xml", "w") as f:
        f.write(response_capacity.text)
else:
    print("❌ Fehler Capacity:", response_capacity.text)

# ==============================================================================
# 2. ACTIVATION PRICE (Hier MUSS es "controlArea_Domain" heißen)
# ==============================================================================
print("\nLade aFRR Arbeitspreise (Energy / CBMP)...")
url_energy = (f"{base_url}&periodStart={start}&periodEnd={end}"
              f"&documentType=A84&processType={processType_aFRR}"
              f"&controlArea_Domain={BiddingZoneDELU}") # <--- controlArea_Domain

response_energy = requests.request("GET", url_energy, headers=headers, data=payload)

if response_energy.status_code == 200:
    print("✅ Energy erfolgreich! (Größe:", len(response_energy.text), "Bytes)")
    with open("afrr_energy_cbmp_2022.xml", "w") as f:
        f.write(response_energy.text)
else:
    print("❌ Fehler Energy:", response_energy.text)

Lade aFRR Leistungspreise (Capacity)...
✅ Capacity erfolgreich!

Lade aFRR Arbeitspreise (Energy / CBMP)...
❌ Fehler Energy: <?xml version="1.0" encoding="UTF-8"?>
<Acknowledgement_MarketDocument
	xmlns="urn:iec62325.351:tc57wg16:451-1:acknowledgementdocument:7:0">
	<mRID>bc4bb3f9-647a-4</mRID>
	<createdDateTime>2026-01-23T21:26:40Z</createdDateTime>
	<sender_MarketParticipant.mRID codingScheme="A01">10X1001A1001A450</sender_MarketParticipant.mRID>
	<sender_MarketParticipant.marketRole.type>A32</sender_MarketParticipant.marketRole.type>
	<receiver_MarketParticipant.mRID codingScheme="A01">10X1001A1001A450</receiver_MarketParticipant.mRID>
	<receiver_MarketParticipant.marketRole.type>A39</receiver_MarketParticipant.marketRole.type>
	<received_MarketDocument.createdDateTime>2026-01-23T21:26:40Z</received_MarketDocument.createdDateTime>
	<Reason>
		<code>999</code>
		<text>The provided parameters do not match the dataItem parameters or values.</text>
	</Reason>
</Acknowledgement_MarketDoc